<a href="https://colab.research.google.com/github/ml-project-sv/Cross-Platform-Binary-Code-Similarity-Detection/blob/main/model_experiment_aggregate-feature-cosine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os, glob, json
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score

In [2]:
PATH = '/content/Cross-Platform-Binary-Code-Similarity-Detection'

In [10]:
def load_dataset(dirs, min_blocks=4, n_feat=8):
  if isinstance(dirs, str):
    dirs = [dirs]

  files = []
  for d in dirs:
    files += glob.glob(os.path.join(d, '*.json'))

  if not files:
    raise FileNotFoundError(f'no .json under {dirs}')

  funcs, fnames, dropped = [], [], 0
  for fp in files:
    print(fp)
    with open(fp) as fh:
      for line in fh:
        line = line.strip()
        if not line: continue

        r = json.loads(line)
        feats = r['features']
        if len(feats) < min_blocks:
          dropped += 1
          continue

        r['X'] = np.asarray(feats, dtype=np.float32)[:, :n_feat]
        r['succs'] = r['succs'] if 'succs' in r else None
        funcs.append(r)
        fnames.append(r['fname'])

  uniq = sorted(set(fnames))
  classes = {n: i for i, n in enumerate(uniq)}
  labels = np.array([classes[n] for n in fnames], dtype=np.int64)
  print(f'[load] files={len(files)} funcs={len(funcs)} classes={len(uniq)} dropped(<{min_blocks} blocks)={dropped} n_feat={n_feat}')

  return funcs, labels, classes


In [12]:
def make_split(labels, ratios=(.8, .1, .1), seed=1337):
  n_cls = labels.max() + 1
  rng = np.random.default_rng(seed)
  cls_perm = rng.permutation(n_cls)

  n_train = int(ratios[0] * n_cls)
  n_val   = int(ratios[1] * n_cls)

  train_cls = set(cls_perm[:n_train].tolist())
  val_cls   = set(cls_perm[n_train: n_train + n_val].tolist())
  test_cls  = set(cls_perm[n_train + n_val:].tolist())


  train_mask = np.array([c in train_cls for c in labels])
  val_mask   = np.array([c in val_cls   for c in labels])
  test_mask  = np.array([c in test_cls  for c in labels])

  print(f'[split] train={train_mask.sum()} val={val_mask.sum()} test={test_mask.sum()}')
  return train_mask, val_mask, test_mask

In [13]:
class BaselineFeatureAggregator():
  def embed_batch(self, funcs):
    out = []
    for r in funcs:
      X = r['X']
      out.append(np.concatenate([X.mean(0), X.sum(0), X.max(0), X.std(0), [len(X)]]))
    E = np.asarray(out, dtype=np.float64)
    return (E - E.mean(0)) / (E.std(0) + 1e-8)

In [11]:
funcs, labels, classes = load_dataset(f'{PATH}/data_acfg/openssl_acfg')


/content/Cross-Platform-Binary-Code-Similarity-Detection/data_acfg/openssl_acfg/openssl_1_0_1f-mips32-O3.json
/content/Cross-Platform-Binary-Code-Similarity-Detection/data_acfg/openssl_acfg/openssl_1_0_1f-arm32-O1.json
/content/Cross-Platform-Binary-Code-Similarity-Detection/data_acfg/openssl_acfg/openssl_1_0_1f-arm32-O2.json
/content/Cross-Platform-Binary-Code-Similarity-Detection/data_acfg/openssl_acfg/openssl_1_0_1f-x86-O1.json
/content/Cross-Platform-Binary-Code-Similarity-Detection/data_acfg/openssl_acfg/openssl_1_0_1f-arm32-O3.json
/content/Cross-Platform-Binary-Code-Similarity-Detection/data_acfg/openssl_acfg/openssl_1_0_1f-x86-O3.json
/content/Cross-Platform-Binary-Code-Similarity-Detection/data_acfg/openssl_acfg/openssl_1_0_1f-mips32-O0.json
/content/Cross-Platform-Binary-Code-Similarity-Detection/data_acfg/openssl_acfg/openssl_1_0_1f-x86-O2.json
/content/Cross-Platform-Binary-Code-Similarity-Detection/data_acfg/openssl_acfg/openssl_1_0_1f-arm32-O0.json
/content/Cross-Platform